In [3]:
import numpy as np
from dataclasses import dataclass
from typing import Callable, Tuple, Optional, Dict, Any, List
from scipy.stats import beta, uniform, lognorm, expon, rayleigh, weibull_min
from scipy.stats import qmc
from scipy.stats import wilcoxon
from itertools import combinations

# =========================================================
# CONFIG
# =========================================================

@dataclass
class SGuAConfig:
    mode: str = "continuous"
    pop_size: int = 50
    max_iters: int = 500
    minimize: bool = True
    sowing_method: str = "random_init"
    mating_model: str = "exponential"
    branching_model: str = "linear"
    vaccination_rate: float = 0.25
    branching_rate: float = 0.5
    mating_pairs: str = "random"
    alpha_mating: float = 4.0
    branch_sigma: float = 0.15
    vacc_intensity: float = 0.5
    seed: Optional[int] = None
    f_target: Optional[float] = None

# =========================================================
# SGuA
# =========================================================
SOWING_METHODS = [
    "random_init",
    "beta_init_3_2",
    "beta_init_2_5_2_5",
    "uniform_init",
    "lognorm_init_0_0_5",
    "lognorm_init_0_0_67",
    "expon_init_0_5",
    "rayleigh_init_0_4",
    "weibull_init_1_1",
    "lhs_init"
]
class SGuA:

    def __init__(self, objective, dim, bounds, config: SGuAConfig):
        self.obj = objective
        self.dim = dim
        self.lb = bounds[:, 0]
        self.ub = bounds[:, 1]
        self.cfg = config
        self.rng = np.random.default_rng(config.seed)

        self.best_x = None
        self.best_f = np.inf
        self.history = []
        self.FE = 0

    # ---------------- Utilities ----------------
    def _evaluate_pop(self, X):
        self.FE += len(X)
        return np.array([self.obj(x) for x in X])

    def _clip(self, X):
        return np.clip(X, self.lb, self.ub)

    def _normalize_to_bounds(self, X):
        X = (X - X.min()) / (X.max() - X.min() + 1e-12)
        return self.lb + X * (self.ub - self.lb)

    def _distance(self, a, b):
        span = np.maximum(self.ub - self.lb, 1e-12)
        d = np.linalg.norm((a - b) / span)
        return min(d / np.sqrt(self.dim), 1.0)

    # ---------------- Sowing ----------------

    def _sowing(self):
        n, d = self.cfg.pop_size, self.dim

        if self.cfg.sowing_method == "random_init":
            X = self.rng.random((n, d))
        elif self.cfg.sowing_method == "beta_init_3_2":
            X = beta.rvs(3, 2, size=(n, d), random_state=self.rng)
        elif self.cfg.sowing_method == "beta_init_2_5_2_5":
            X = beta.rvs(2.5, 2.5, size=(n, d), random_state=self.rng)
        elif self.cfg.sowing_method == "uniform_init":
            X = uniform.rvs(size=(n, d), random_state=self.rng)
        elif self.cfg.sowing_method == "lognorm_init_0_0_5":
            X = lognorm.rvs(s=0.5, size=(n, d), random_state=self.rng)
        elif self.cfg.sowing_method == "lognorm_init_0_0_67":
            X = lognorm.rvs(s=0.67, size=(n, d), random_state=self.rng)
        elif self.cfg.sowing_method == "expon_init_0_5":
            X = expon.rvs(scale=2.0, size=(n, d), random_state=self.rng)
        elif self.cfg.sowing_method == "rayleigh_init_0_4":
            X = rayleigh.rvs(scale=0.4, size=(n, d), random_state=self.rng)
        elif self.cfg.sowing_method == "weibull_init_1_1":
            X = weibull_min.rvs(1, scale=1, size=(n, d), random_state=self.rng)
        elif self.cfg.sowing_method == "lhs_init":
            sampler = qmc.LatinHypercube(d=d, seed=self.rng)
            X = sampler.random(n)
        else:
            raise ValueError("Unknown sowing method")

        return self._normalize_to_bounds(X)

    # ---------------- Operators ----------------

    def _mating(self, G):
        idx = self.rng.permutation(len(G))
        children = []

        for i in range(0, len(idx) - 1, 2):
            a, b = G[idx[i]], G[idx[i + 1]]
            d = self._distance(a, b)
            p = np.exp(-self.cfg.alpha_mating * d)
            if self.rng.random() <= p:
                w = self.rng.random(self.dim)
                children.append(self._clip(w * a + (1 - w) * b))

        return np.array(children)

    def _branching(self, G):
        n = len(G)
        m = max(1, int(self.cfg.branching_rate * n))
        idxs = self.rng.choice(n, size=m, replace=False)
        kids = []

        for i in idxs:
            x = G[i].copy()
            j = self.rng.integers(0, self.dim)
            span = self.ub[j] - self.lb[j]
            x[j] += self.rng.normal(0, self.cfg.branch_sigma * span)
            kids.append(self._clip(x))

        return np.array(kids)

    def _vaccinating(self, G, F):
        n = len(G)
        m = max(1, int(self.cfg.vaccination_rate * n))
        order = np.argsort(F)
        best = order[:n // 2]
        worst = order[n // 2:]
        kids = []

        for i in self.rng.choice(worst, size=m, replace=False):
            donor = G[self.rng.choice(best)]
            kids.append(self._clip(G[i] + self.cfg.vacc_intensity * (donor - G[i])))

        return np.array(kids)

    # ---------------- Run ----------------

    def run(self):
        G = self._sowing()
        F = self._evaluate_pop(G)

        self.best_f = F.min()
        self.best_x = G[F.argmin()].copy()
        self.history = [self.best_f]

        for _ in range(self.cfg.max_iters):
            G_all = np.vstack([G, self._mating(G), self._branching(G), self._vaccinating(G, F)])
            F_all = self._evaluate_pop(G_all)
            order = np.argsort(F_all)
            G = G_all[order[:self.cfg.pop_size]]
            F = F_all[order[:self.cfg.pop_size]]

            if F[0] < self.best_f:
                self.best_f = F[0]
                self.best_x = G[0].copy()

            self.history.append(self.best_f)

        return {"best_x": self.best_x, "best_f": self.best_f, "history": np.array(self.history),"FE": self.FE}

# =========================================================
# CEC FUNCTIONS (F1–F11)
# =========================================================
def u(x, a, k, m):
    return np.where(
        x > a, k * (x - a)**m,
        np.where(x < -a, k * (-x - a)**m, 0)
    )
def get_cec_function_details(D=10):
    functions = {}

    functions["F1"] = (lambda x: np.sum(x**2),
        np.column_stack([np.full(D, -100.0), np.full(D, 100.0)]))

    functions["F2"] = (lambda x: np.sum(np.abs(x)) + np.prod(np.abs(x)),
        np.column_stack([np.full(D, -10.0), np.full(D, 10.0)]))

    functions["F3"] = (lambda x: sum((np.sum(x[:i+1]))**2 for i in range(len(x))),
        np.column_stack([np.full(D, -100.0), np.full(D, 100.0)]))

    functions["F4"] = (lambda x: np.max(np.abs(x)),
        np.column_stack([np.full(D, -100.0), np.full(D, 100.0)]))

    functions["F5"] = (lambda x: np.sum(100*(x[1:] - x[:-1]**2)**2 + (x[:-1] - 1)**2),
        np.column_stack([np.full(D, -30.0), np.full(D, 30.0)]))

    functions["F6"] = (lambda x: np.sum((np.floor(x) + 0.5)**2),
        np.column_stack([np.full(D, -100.0), np.full(D, 100.0)]))

    functions["F7"] = (lambda x: np.sum(np.arange(1, len(x)+1) * x**4),
        np.column_stack([np.full(D, -1.28), np.full(D, 1.28)]))

    functions["F8"] = (lambda x: -np.sum(x * np.sin(np.sqrt(np.abs(x)))),
        np.column_stack([np.full(D, -500.0), np.full(D, 500.0)]))

    functions["F9"] = (lambda x: np.sum(x**2 - 10*np.cos(2*np.pi*x) + 10),
        np.column_stack([np.full(D, -5.12), np.full(D, 5.12)]))

    functions["F10"] = (lambda x: -20*np.exp(-0.2*np.sqrt(np.mean(x**2)))
        - np.exp(np.mean(np.cos(2*np.pi*x))) + 20 + np.e,
        np.column_stack([np.full(D, -32.0), np.full(D, 32.0)]))

    functions["F11"] = (lambda x: np.sum(x**2)/4000 - np.prod(np.cos(x/np.sqrt(np.arange(1,len(x)+1)))) + 1,
        np.column_stack([np.full(D, -600.0), np.full(D, 600.0)]))
    # Penalized 1
    def penalized_1(x):
        n = len(x)
        y = 1 + (x + 1) / 4
        term1 = np.pi / n * (
            10 * np.sin(np.pi * y[0])**2 +
            np.sum((y[:-1] - 1)**2 * (1 + 10 * np.sin(np.pi * y[1:])**2)) +
            (y[-1] - 1)**2
        )
        term2 = np.sum(u(x, 10, 100, 4))
        return term1 + term2

    bounds_p1 = np.column_stack([np.full(D, -50.0), np.full(D, 50.0)])
    functions["F12: Penalized 1"] = (penalized_1, bounds_p1)

    # Penalized 2
    def penalized_2(x):
        n = len(x)
        term1 = 0.1 * (
            np.sin(3 * np.pi * x[0])**2 +
            np.sum((x[:-1] - 1)**2 * (1 + np.sin(3 * np.pi * x[1:])**2)) +
            (x[-1] - 1)**2 * (1 + np.sin(2 * np.pi * x[-1])**2)
        )
        term2 = np.sum(u(x, 5, 100, 4))
        return term1 + term2

    bounds_p2 = np.column_stack([np.full(D, -50.0), np.full(D, 50.0)])
    functions["F13: Penalized 2"] = (penalized_2, bounds_p2)
    return functions

def get_fixed_cec_functions():
    functions = {}

    # ======================================================
    # Shekel's Foxholes (2D)
    # ======================================================
    def shekel(x):
        a = np.array([
            [-32, -16, 0, 16, 32] * 5,
            [-32]*5 + [-16]*5 + [0]*5 + [16]*5 + [32]*5
        ])
        c = np.arange(1, 26)
        return np.sum(1.0 / (c + np.sum((x[:, None] - a) ** 6, axis=0)))

    bounds = np.array([[-65, 65], [-65, 65]])
    functions["Shekel"] = (shekel, bounds, 2)

    # ======================================================
    # Branin (2D)
    # ======================================================
    def branin(x):
        x1, x2 = x
        a = 1.0
        b = 5.1 / (4 * np.pi**2)
        c = 5 / np.pi
        r = 6
        s = 10
        t = 1 / (8 * np.pi)
        return a * (x2 - b*x1**2 + c*x1 - r)**2 + s*(1 - t)*np.cos(x1) + s

    bounds = np.array([[-5, 5], [0, 15]])
    functions["Branin"] = (branin, bounds, 2)

    # ======================================================
    # Six-Hump Camel (2D)
    # ======================================================
    def six_hump_camel(x):
        x1, x2 = x
        return (
            (4 - 2.1*x1**2 + (x1**4)/3)*x1**2 +
            x1*x2 +
            (-4 + 4*x2**2)*x2**2
        )

    bounds = np.array([[-5, 5], [-5, 5]])
    functions["Six-Hump Camel"] = (six_hump_camel, bounds, 2)

    # ======================================================
    # Goldstein-Price (2D)
    # ======================================================
    def goldstein_price(x):
        x1, x2 = x
        term1 = 1 + (x1 + x2 + 1)**2 * (
            19 - 14*x1 + 3*x1**2 - 14*x2 + 6*x1*x2 + 3*x2**2
        )
        term2 = 30 + (2*x1 - 3*x2)**2 * (
            18 - 32*x1 + 12*x1**2 + 48*x2 - 36*x1*x2 + 27*x2**2
        )
        return term1 * term2

    bounds = np.array([[-2, 2], [-2, 2]])
    functions["Goldstein-Price"] = (goldstein_price, bounds, 2)

    # ======================================================
    # Kowalik's Function (4D)
    # ======================================================
    def kowalik(x):
        a = np.array([
            0.1957, 0.1947, 0.1735, 0.16,
            0.0844, 0.0627, 0.0456, 0.0342,
            0.0323, 0.0235, 0.0246
        ])
        b = 1.0 / np.array([
            0.25, 0.5, 1, 2, 4, 6,
            8, 10, 12, 14, 16
        ])
        return np.sum((a - (x[0]*(b**2 + b*x[1]) /
                (b**2 + b*x[2] + x[3])))**2)

    bounds = np.array([[-5, 5]] * 4)
    functions["Kowalik"] = (kowalik, bounds, 4)

    # ======================================================
    # Hartman 3D
    # ======================================================
    def hartman3(x):
        alpha = np.array([1.0, 1.2, 3.0, 3.2])
        A = np.array([
            [3.0, 10, 30],
            [0.1, 10, 35],
            [3.0, 10, 30],
            [0.1, 10, 35]
        ])
        P = 1e-4 * np.array([
            [3689, 1170, 2673],
            [4699, 4387, 7470],
            [1091, 8732, 5547],
            [381, 5743, 8828]
        ])
        return -np.sum(alpha * np.exp(-np.sum(A * (x - P)**2, axis=1)))

    bounds = np.array([[0, 1]] * 3)
    functions["Hartman 3D"] = (hartman3, bounds, 3)

    # ======================================================
    # Hartman 6D
    # ======================================================
    def hartman6(x):
        alpha = np.array([1.0, 1.2, 3.0, 3.2])
        A = np.array([
            [10, 3, 17, 3.5, 1.7, 8],
            [0.05, 10, 17, 0.1, 8, 14],
            [3, 3.5, 1.7, 10, 17, 8],
            [17, 8, 0.05, 10, 0.1, 14]
        ])
        P = 1e-4 * np.array([
            [1312, 1696, 5569, 124, 8283, 5886],
            [2329, 4135, 8307, 3736, 1004, 9991],
            [2348, 1451, 3522, 2883, 3047, 6650],
            [4047, 8828, 8732, 5743, 1091, 381]
        ])
        return -np.sum(alpha * np.exp(-np.sum(A * (x - P)**2, axis=1)))

    bounds = np.array([[0, 1]] * 6)
    functions["Hartman 6D"] = (hartman6, bounds, 6)

    return functions
# =========================================================
# EXPERIMENT RUNNER
# =========================================================
def run_fixed_cec_tests(N_RUNS=30, MAX_ITERS=500, POP_SIZE=50):

    functions = get_fixed_cec_functions()

    print("\n=== Fixed-Dimension CEC Functions ===")
    print("Function        Sowing              Best        Mean        Std")

    for fname, (fn, bounds, dim) in functions.items():

        for sowing_method in SOWING_METHODS:

            scores = []

            for r in range(N_RUNS):
                cfg = SGuAConfig(
                    pop_size=POP_SIZE,
                    max_iters=MAX_ITERS,
                    sowing_method=sowing_method,
                    seed=r
                )

                algo = SGuA(fn, dim, bounds, cfg)
                res = algo.run()
                scores.append(res["best_f"])

            scores = np.array(scores)

            print(
                f"{fname:<15} "
                f"{sowing_method:<18} "
                f"{scores.min():.3e} "
                f"{scores.mean():.3e} "
                f"{scores.std():.3e}"
            )

def run_cec_tests(D=10, N_RUNS=30, MAX_ITERS=500, POP_SIZE=50, sowing_methods=None):

    if sowing_methods is None:
        sowing_methods = ["random_init", "beta_init_3_2", "lhs_init"]

    functions = get_cec_function_details(D)

    print("Function  Sowing              Best        Mean        Std         FE")

    for name, (fn, bounds) in functions.items():

        all_scores = {}   # sowing -> scores
        all_fes = {}

        for sowing in sowing_methods:

            cfg = SGuAConfig(
                pop_size=POP_SIZE,
                max_iters=MAX_ITERS,
                sowing_method=sowing
            )

            scores = []
            fes = []

            for r in range(N_RUNS):
                cfg.seed = r
                algo = SGuA(fn, D, bounds, cfg)
                res = algo.run()
                scores.append(res["best_f"])
                fes.append(res["FE"])

            scores = np.array(scores)
            all_scores[sowing] = scores
            all_fes[sowing] = np.mean(fes)

            print(
                f"{name:<8} "
                f"{sowing:<18} "
                f"{scores.min():.3e} "
                f"{scores.mean():.3e} "
                f"{scores.std():.3e} "
                f"{all_fes[sowing]:.0f}"
            )

        # ===== WILCOXON TAM BURADA =====
        print(f"\nWilcoxon results for {name}:")
        pairs = list(combinations(comparison_sowing_methods, 2))

        # Only perform Wilcoxon tests if all methods for comparison were run
        if all(method in all_scores for method in [p for pair in pairs for p in pair]):
            for a, b in pairs:
                stat, p = wilcoxon(all_scores[a], all_scores[b])
                print(f"  {a} vs {b} : p = {p:.4f}")
        else:
            print("  Not enough sowing methods were run to perform all Wilcoxon comparisons.")

        print("-" * 70)
# =========================================================
# MAIN
# =========================================================

if __name__ == "__main__":

    DIMENSION = 10
    RUNS = 30
    ITERS = 500
    POP = 50

    # Run CEC tests with all comparison methods in one go
    comparison_sowing_methods = [
         "random_init",
        "beta_init_3_2",
        "beta_init_2_5_2_5",
        "uniform_init",
        "lognorm_init_0_0_5",
        "lognorm_init_0_0_67",
        "expon_init_0_5",
        "rayleigh_init_0_4",
        "weibull_init_1_1",
        "lhs_init"
    ]
    print(f"\n=== Running CEC tests with multiple sowing methods for comparison ===")
    run_cec_tests(
        D=DIMENSION,
        N_RUNS=RUNS,
        MAX_ITERS=ITERS,
        POP_SIZE=POP,
        sowing_methods=comparison_sowing_methods
    )

    print("\n=== Fixed-dimension CEC tests ===")
    run_fixed_cec_tests(
        N_RUNS=RUNS,
        MAX_ITERS=ITERS,
        POP_SIZE=POP
    )


=== Running CEC tests with multiple sowing methods for comparison ===
Function  Sowing              Best        Mean        Std         FE
F1       random_init        1.338e-03 1.308e-02 1.007e-02 55902
F1       beta_init_3_2      8.606e-04 2.057e-02 1.830e-02 55909
F1       beta_init_2_5_2_5  6.137e-04 1.318e-02 1.138e-02 55929
F1       uniform_init       1.338e-03 1.308e-02 1.007e-02 55902
F1       lognorm_init_0_0_5 1.004e-04 2.204e-02 1.722e-02 55773
F1       lognorm_init_0_0_67 7.163e-04 2.263e-02 1.725e-02 55713
F1       expon_init_0_5     4.314e-03 3.253e-02 3.707e-02 55703
F1       rayleigh_init_0_4  2.447e-03 2.251e-02 2.094e-02 55871
F1       weibull_init_1_1   2.281e-03 3.716e-02 3.283e-02 55709
F1       lhs_init           1.878e-03 1.603e-02 1.149e-02 55904

Wilcoxon results for F1:
  random_init vs beta_init_3_2 : p = 0.1981
  random_init vs beta_init_2_5_2_5 : p = 0.6263
  random_init vs uniform_init : p = nan
  random_init vs lognorm_init_0_0_5 : p = 0.0113
  random_ini

d=30

In [5]:
import numpy as np
from dataclasses import dataclass
from typing import Callable, Tuple, Optional, Dict, Any, List
from scipy.stats import beta, uniform, lognorm, expon, rayleigh, weibull_min
from scipy.stats import qmc
from scipy.stats import wilcoxon
from itertools import combinations

# =========================================================
# CONFIG
# =========================================================

@dataclass
class SGuAConfig:
    mode: str = "continuous"
    pop_size: int = 50
    max_iters: int = 500
    minimize: bool = True
    sowing_method: str = "random_init"
    mating_model: str = "exponential"
    branching_model: str = "linear"
    vaccination_rate: float = 0.25
    branching_rate: float = 0.5
    mating_pairs: str = "random"
    alpha_mating: float = 4.0
    branch_sigma: float = 0.15
    vacc_intensity: float = 0.5
    seed: Optional[int] = None
    f_target: Optional[float] = None

# =========================================================
# SGuA
# =========================================================
SOWING_METHODS = [
    "random_init",
    "beta_init_3_2",
    "beta_init_2_5_2_5",
    "uniform_init",
    "lognorm_init_0_0_5",
    "lognorm_init_0_0_67",
    "expon_init_0_5",
    "rayleigh_init_0_4",
    "weibull_init_1_1",
    "lhs_init"
]
class SGuA:

    def __init__(self, objective, dim, bounds, config: SGuAConfig):
        self.obj = objective
        self.dim = dim
        self.lb = bounds[:, 0]
        self.ub = bounds[:, 1]
        self.cfg = config
        self.rng = np.random.default_rng(config.seed)

        self.best_x = None
        self.best_f = np.inf
        self.history = []
        self.FE = 0

    # ---------------- Utilities ----------------
    def _evaluate_pop(self, X):
        self.FE += len(X)
        return np.array([self.obj(x) for x in X])

    def _clip(self, X):
        return np.clip(X, self.lb, self.ub)

    def _normalize_to_bounds(self, X):
        X = (X - X.min()) / (X.max() - X.min() + 1e-12)
        return self.lb + X * (self.ub - self.lb)

    def _distance(self, a, b):
        span = np.maximum(self.ub - self.lb, 1e-12)
        d = np.linalg.norm((a - b) / span)
        return min(d / np.sqrt(self.dim), 1.0)

    # ---------------- Sowing ----------------

    def _sowing(self):
        n, d = self.cfg.pop_size, self.dim

        if self.cfg.sowing_method == "random_init":
            X = self.rng.random((n, d))
        elif self.cfg.sowing_method == "beta_init_3_2":
            X = beta.rvs(3, 2, size=(n, d), random_state=self.rng)
        elif self.cfg.sowing_method == "beta_init_2_5_2_5":
            X = beta.rvs(2.5, 2.5, size=(n, d), random_state=self.rng)
        elif self.cfg.sowing_method == "uniform_init":
            X = uniform.rvs(size=(n, d), random_state=self.rng)
        elif self.cfg.sowing_method == "lognorm_init_0_0_5":
            X = lognorm.rvs(s=0.5, size=(n, d), random_state=self.rng)
        elif self.cfg.sowing_method == "lognorm_init_0_0_67":
            X = lognorm.rvs(s=0.67, size=(n, d), random_state=self.rng)
        elif self.cfg.sowing_method == "expon_init_0_5":
            X = expon.rvs(scale=2.0, size=(n, d), random_state=self.rng)
        elif self.cfg.sowing_method == "rayleigh_init_0_4":
            X = rayleigh.rvs(scale=0.4, size=(n, d), random_state=self.rng)
        elif self.cfg.sowing_method == "weibull_init_1_1":
            X = weibull_min.rvs(1, scale=1, size=(n, d), random_state=self.rng)
        elif self.cfg.sowing_method == "lhs_init":
            sampler = qmc.LatinHypercube(d=d, seed=self.rng)
            X = sampler.random(n)
        else:
            raise ValueError("Unknown sowing method")

        return self._normalize_to_bounds(X)

    # ---------------- Operators ----------------

    def _mating(self, G):
        idx = self.rng.permutation(len(G))
        children = []

        for i in range(0, len(idx) - 1, 2):
            a, b = G[idx[i]], G[idx[i + 1]]
            d = self._distance(a, b)
            p = np.exp(-self.cfg.alpha_mating * d)
            if self.rng.random() <= p:
                w = self.rng.random(self.dim)
                children.append(self._clip(w * a + (1 - w) * b))

        children = np.array(children)
        # **YENİ KONTROL EKLE:** Boşsa, 0 satırlı, D sütunlu bir dizi döndür.
        if children.size == 0:
            return np.empty((0, self.dim))
    
        return children

    def _branching(self, G):
        n = len(G)
        m = max(1, int(self.cfg.branching_rate * n))
        idxs = self.rng.choice(n, size=m, replace=False)
        kids = []

        for i in idxs:
            x = G[i].copy()
            j = self.rng.integers(0, self.dim)
            span = self.ub[j] - self.lb[j]
            x[j] += self.rng.normal(0, self.cfg.branch_sigma * span)
            kids.append(self._clip(x))

        kids = np.array(kids)
        # **YENİ KONTROL EKLE:**
        if kids.size == 0:
            return np.empty((0, self.dim))
        
        return kids

    def _vaccinating(self, G, F):
        n = len(G)
        m = max(1, int(self.cfg.vaccination_rate * n))
        order = np.argsort(F)
        best = order[:n // 2]
        worst = order[n // 2:]
        kids = []

        for i in self.rng.choice(worst, size=m, replace=False):
            donor = G[self.rng.choice(best)]
            kids.append(self._clip(G[i] + self.cfg.vacc_intensity * (donor - G[i])))

        kids = np.array(kids)
        # **YENİ KONTROL EKLE:**
        if kids.size == 0:
            return np.empty((0, self.dim))
        
        return kids

    # ---------------- Run ----------------

    def run(self):
        G = self._sowing()
        F = self._evaluate_pop(G)

        self.best_f = F.min()
        self.best_x = G[F.argmin()].copy()
        self.history = [self.best_f]

        for _ in range(self.cfg.max_iters):
            G_all = np.vstack([G, self._mating(G), self._branching(G), self._vaccinating(G, F)])
            F_all = self._evaluate_pop(G_all)
            order = np.argsort(F_all)
            G = G_all[order[:self.cfg.pop_size]]
            F = F_all[order[:self.cfg.pop_size]]

            if F[0] < self.best_f:
                self.best_f = F[0]
                self.best_x = G[0].copy()

            self.history.append(self.best_f)

        return {"best_x": self.best_x, "best_f": self.best_f, "history": np.array(self.history),"FE": self.FE}

# =========================================================
# CEC FUNCTIONS (F1–F11)
# =========================================================
def f3_optimized(x):
    # np.cumsum(x): x'in kümülatif toplamını tek bir hızlı NumPy işlemiyle hesaplar.
    # Örneğin, [a, b, c] -> [a, a+b, a+b+c]
    cumulative_sums = np.cumsum(x)
    
    # Kümülatif toplamların karesini alıp toplar.
    return np.sum(cumulative_sums**2)
    
def u(x, a, k, m):
    return np.where(
        x > a, k * (x - a)**m,
        np.where(x < -a, k * (-x - a)**m, 0)
    )
def get_cec_function_details(D=10):
    functions = {}

    functions["F1"] = (lambda x: np.sum(x**2),
        np.column_stack([np.full(D, -100.0), np.full(D, 100.0)]))

    functions["F2"] = (lambda x: np.sum(np.abs(x)) + np.prod(np.abs(x)),
        np.column_stack([np.full(D, -10.0), np.full(D, 10.0)]))

    functions["F3"] = (
        f3_optimized,
        np.column_stack([np.full(D, -100.0), np.full(D, 100.0)]))

    functions["F4"] = (lambda x: np.max(np.abs(x)),
        np.column_stack([np.full(D, -100.0), np.full(D, 100.0)]))

    functions["F5"] = (lambda x: np.sum(100*(x[1:] - x[:-1]**2)**2 + (x[:-1] - 1)**2),
        np.column_stack([np.full(D, -30.0), np.full(D, 30.0)]))

    functions["F6"] = (lambda x: np.sum((np.floor(x) + 0.5)**2),
        np.column_stack([np.full(D, -100.0), np.full(D, 100.0)]))

    functions["F7"] = (lambda x: np.sum(np.arange(1, len(x)+1) * x**4),
        np.column_stack([np.full(D, -1.28), np.full(D, 1.28)]))

    functions["F8"] = (lambda x: -np.sum(x * np.sin(np.sqrt(np.abs(x)))),
        np.column_stack([np.full(D, -500.0), np.full(D, 500.0)]))

    functions["F9"] = (lambda x: np.sum(x**2 - 10*np.cos(2*np.pi*x) + 10),
        np.column_stack([np.full(D, -5.12), np.full(D, 5.12)]))

    functions["F10"] = (lambda x: -20*np.exp(-0.2*np.sqrt(np.mean(x**2)))
        - np.exp(np.mean(np.cos(2*np.pi*x))) + 20 + np.e,
        np.column_stack([np.full(D, -32.0), np.full(D, 32.0)]))

    functions["F11"] = (lambda x: np.sum(x**2)/4000 - np.prod(np.cos(x/np.sqrt(np.arange(1,len(x)+1)))) + 1,
        np.column_stack([np.full(D, -600.0), np.full(D, 600.0)]))
    # Penalized 1
    def penalized_1(x):
        n = len(x)
        y = 1 + (x + 1) / 4
        term1 = np.pi / n * (
            10 * np.sin(np.pi * y[0])**2 +
            np.sum((y[:-1] - 1)**2 * (1 + 10 * np.sin(np.pi * y[1:])**2)) +
            (y[-1] - 1)**2
        )
        term2 = np.sum(u(x, 10, 100, 4))
        return term1 + term2

    bounds_p1 = np.column_stack([np.full(D, -50.0), np.full(D, 50.0)])
    functions["F12: Penalized 1"] = (penalized_1, bounds_p1)

    # Penalized 2
    def penalized_2(x):
        n = len(x)
        term1 = 0.1 * (
            np.sin(3 * np.pi * x[0])**2 +
            np.sum((x[:-1] - 1)**2 * (1 + np.sin(3 * np.pi * x[1:])**2)) +
            (x[-1] - 1)**2 * (1 + np.sin(2 * np.pi * x[-1])**2)
        )
        term2 = np.sum(u(x, 5, 100, 4))
        return term1 + term2

    bounds_p2 = np.column_stack([np.full(D, -50.0), np.full(D, 50.0)])
    functions["F13: Penalized 2"] = (penalized_2, bounds_p2)
    return functions

def get_fixed_cec_functions():
    functions = {}

    # ======================================================
    # Shekel's Foxholes (2D)
    # ======================================================
    def shekel(x):
        a = np.array([
            [-32, -16, 0, 16, 32] * 5,
            [-32]*5 + [-16]*5 + [0]*5 + [16]*5 + [32]*5
        ])
        c = np.arange(1, 26)
        return np.sum(1.0 / (c + np.sum((x[:, None] - a) ** 6, axis=0)))

    bounds = np.array([[-65, 65], [-65, 65]])
    functions["Shekel"] = (shekel, bounds, 2)

    # ======================================================
    # Branin (2D)
    # ======================================================
    def branin(x):
        x1, x2 = x
        a = 1.0
        b = 5.1 / (4 * np.pi**2)
        c = 5 / np.pi
        r = 6
        s = 10
        t = 1 / (8 * np.pi)
        return a * (x2 - b*x1**2 + c*x1 - r)**2 + s*(1 - t)*np.cos(x1) + s

    bounds = np.array([[-5, 5], [0, 15]])
    functions["Branin"] = (branin, bounds, 2)

    # ======================================================
    # Six-Hump Camel (2D)
    # ======================================================
    def six_hump_camel(x):
        x1, x2 = x
        return (
            (4 - 2.1*x1**2 + (x1**4)/3)*x1**2 +
            x1*x2 +
            (-4 + 4*x2**2)*x2**2
        )

    bounds = np.array([[-5, 5], [-5, 5]])
    functions["Six-Hump Camel"] = (six_hump_camel, bounds, 2)

    # ======================================================
    # Goldstein-Price (2D)
    # ======================================================
    def goldstein_price(x):
        x1, x2 = x
        term1 = 1 + (x1 + x2 + 1)**2 * (
            19 - 14*x1 + 3*x1**2 - 14*x2 + 6*x1*x2 + 3*x2**2
        )
        term2 = 30 + (2*x1 - 3*x2)**2 * (
            18 - 32*x1 + 12*x1**2 + 48*x2 - 36*x1*x2 + 27*x2**2
        )
        return term1 * term2

    bounds = np.array([[-2, 2], [-2, 2]])
    functions["Goldstein-Price"] = (goldstein_price, bounds, 2)

    # ======================================================
    # Kowalik's Function (4D)
    # ======================================================
    def kowalik(x):
        a = np.array([
            0.1957, 0.1947, 0.1735, 0.16,
            0.0844, 0.0627, 0.0456, 0.0342,
            0.0323, 0.0235, 0.0246
        ])
        b = 1.0 / np.array([
            0.25, 0.5, 1, 2, 4, 6,
            8, 10, 12, 14, 16
        ])
        return np.sum((a - (x[0]*(b**2 + b*x[1]) /
                (b**2 + b*x[2] + x[3])))**2)

    bounds = np.array([[-5, 5]] * 4)
    functions["Kowalik"] = (kowalik, bounds, 4)

    # ======================================================
    # Hartman 3D
    # ======================================================
    def hartman3(x):
        alpha = np.array([1.0, 1.2, 3.0, 3.2])
        A = np.array([
            [3.0, 10, 30],
            [0.1, 10, 35],
            [3.0, 10, 30],
            [0.1, 10, 35]
        ])
        P = 1e-4 * np.array([
            [3689, 1170, 2673],
            [4699, 4387, 7470],
            [1091, 8732, 5547],
            [381, 5743, 8828]
        ])
        return -np.sum(alpha * np.exp(-np.sum(A * (x - P)**2, axis=1)))

    bounds = np.array([[0, 1]] * 3)
    functions["Hartman 3D"] = (hartman3, bounds, 3)

    # ======================================================
    # Hartman 6D
    # ======================================================
    def hartman6(x):
        alpha = np.array([1.0, 1.2, 3.0, 3.2])
        A = np.array([
            [10, 3, 17, 3.5, 1.7, 8],
            [0.05, 10, 17, 0.1, 8, 14],
            [3, 3.5, 1.7, 10, 17, 8],
            [17, 8, 0.05, 10, 0.1, 14]
        ])
        P = 1e-4 * np.array([
            [1312, 1696, 5569, 124, 8283, 5886],
            [2329, 4135, 8307, 3736, 1004, 9991],
            [2348, 1451, 3522, 2883, 3047, 6650],
            [4047, 8828, 8732, 5743, 1091, 381]
        ])
        return -np.sum(alpha * np.exp(-np.sum(A * (x - P)**2, axis=1)))

    bounds = np.array([[0, 1]] * 6)
    functions["Hartman 6D"] = (hartman6, bounds, 6)

    return functions
# =========================================================
# EXPERIMENT RUNNER
# =========================================================
def run_fixed_cec_tests(N_RUNS=30, MAX_ITERS=500, POP_SIZE=50):

    functions = get_fixed_cec_functions()

    print("\n=== Fixed-Dimension CEC Functions ===")
    print("Function        Sowing              Best        Mean        Std")

    for fname, (fn, bounds, dim) in functions.items():

        for sowing_method in SOWING_METHODS:

            scores = []

            for r in range(N_RUNS):
                cfg = SGuAConfig(
                    pop_size=POP_SIZE,
                    max_iters=MAX_ITERS,
                    sowing_method=sowing_method,
                    seed=r
                )

                algo = SGuA(fn, dim, bounds, cfg)
                res = algo.run()
                scores.append(res["best_f"])

            scores = np.array(scores)

            print(
                f"{fname:<15} "
                f"{sowing_method:<18} "
                f"{scores.min():.3e} "
                f"{scores.mean():.3e} "
                f"{scores.std():.3e}"
            )

def run_cec_tests(D=10, N_RUNS=30, MAX_ITERS=500, POP_SIZE=50, sowing_methods=None):

    if sowing_methods is None:
        sowing_methods = ["random_init", "beta_init_3_2", "lhs_init"]

    functions = get_cec_function_details(D)

    print("Function  Sowing              Best        Mean        Std         FE")

    for name, (fn, bounds) in functions.items():

        all_scores = {}   # sowing -> scores
        all_fes = {}

        for sowing in sowing_methods:

            cfg = SGuAConfig(
                pop_size=POP_SIZE,
                max_iters=MAX_ITERS,
                sowing_method=sowing
            )

            scores = []
            fes = []

            for r in range(N_RUNS):
                cfg.seed = r
                algo = SGuA(fn, D, bounds, cfg)
                res = algo.run()
                scores.append(res["best_f"])
                fes.append(res["FE"])

            scores = np.array(scores)
            all_scores[sowing] = scores
            all_fes[sowing] = np.mean(fes)

            print(
                f"{name:<8} "
                f"{sowing:<18} "
                f"{scores.min():.3e} "
                f"{scores.mean():.3e} "
                f"{scores.std():.3e} "
                f"{all_fes[sowing]:.0f}"
            )

        # ===== WILCOXON TAM BURADA =====
        print(f"\nWilcoxon results for {name}:")
        pairs = list(combinations(comparison_sowing_methods, 2))

        # Only perform Wilcoxon tests if all methods for comparison were run
        if all(method in all_scores for method in [p for pair in pairs for p in pair]):
            for a, b in pairs:
                stat, p = wilcoxon(all_scores[a], all_scores[b])
                print(f"  {a} vs {b} : p = {p:.4f}")
        else:
            print("  Not enough sowing methods were run to perform all Wilcoxon comparisons.")

        print("-" * 70)
# =========================================================
# MAIN
# =========================================================

if __name__ == "__main__":

    DIMENSION = 30
    RUNS = 30
    ITERS = 1500
    POP = 50

    # Run CEC tests with all comparison methods in one go
    comparison_sowing_methods = [
         "random_init",
        "beta_init_3_2",
        "beta_init_2_5_2_5",
        "uniform_init",
        "lognorm_init_0_0_5",
        "lognorm_init_0_0_67",
        "expon_init_0_5",
        "rayleigh_init_0_4",
        "weibull_init_1_1",
        "lhs_init"
    ]
    print(f"\n=== Running CEC tests with multiple sowing methods for comparison ===")
    run_cec_tests(
        D=DIMENSION,
        N_RUNS=RUNS,
        MAX_ITERS=ITERS,
        POP_SIZE=POP,
        sowing_methods=comparison_sowing_methods
    )

    print("\n=== Fixed-dimension CEC tests ===")
    run_fixed_cec_tests(
        N_RUNS=RUNS,
        MAX_ITERS=ITERS,
        POP_SIZE=POP
    )


=== Running CEC tests with multiple sowing methods for comparison ===
Function  Sowing              Best        Mean        Std         FE
F1       random_init        2.620e-02 6.928e-02 3.601e-02 167840
F1       beta_init_3_2      2.641e-02 7.683e-02 4.506e-02 167819
F1       beta_init_2_5_2_5  2.278e-02 7.275e-02 3.290e-02 167888
F1       uniform_init       2.620e-02 6.928e-02 3.601e-02 167840
F1       lognorm_init_0_0_5 2.727e-02 1.069e-01 6.465e-02 167447
F1       lognorm_init_0_0_67 4.958e-02 1.121e-01 4.208e-02 167331
F1       expon_init_0_5     3.749e-02 1.149e-01 4.972e-02 167338
F1       rayleigh_init_0_4  2.889e-02 8.375e-02 3.288e-02 167667
F1       weibull_init_1_1   4.581e-02 1.171e-01 6.106e-02 167347
F1       lhs_init           7.460e-03 5.623e-02 2.069e-02 167851

Wilcoxon results for F1:
  random_init vs beta_init_3_2 : p = 0.3085
  random_init vs beta_init_2_5_2_5 : p = 0.8078
  random_init vs uniform_init : p = nan
  random_init vs lognorm_init_0_0_5 : p = 0.0087
  

d=50

In [6]:
import numpy as np
from dataclasses import dataclass
from typing import Callable, Tuple, Optional, Dict, Any, List
from scipy.stats import beta, uniform, lognorm, expon, rayleigh, weibull_min
from scipy.stats import qmc
from scipy.stats import wilcoxon
from itertools import combinations

# =========================================================
# CONFIG
# =========================================================

@dataclass
class SGuAConfig:
    mode: str = "continuous"
    pop_size: int = 50
    max_iters: int = 500
    minimize: bool = True
    sowing_method: str = "random_init"
    mating_model: str = "exponential"
    branching_model: str = "linear"
    vaccination_rate: float = 0.25
    branching_rate: float = 0.5
    mating_pairs: str = "random"
    alpha_mating: float = 4.0
    branch_sigma: float = 0.15
    vacc_intensity: float = 0.5
    seed: Optional[int] = None
    f_target: Optional[float] = None

# =========================================================
# SGuA
# =========================================================
SOWING_METHODS = [
    "random_init",
    "beta_init_3_2",
    "beta_init_2_5_2_5",
    "uniform_init",
    "lognorm_init_0_0_5",
    "lognorm_init_0_0_67",
    "expon_init_0_5",
    "rayleigh_init_0_4",
    "weibull_init_1_1",
    "lhs_init"
]
class SGuA:

    def __init__(self, objective, dim, bounds, config: SGuAConfig):
        self.obj = objective
        self.dim = dim
        self.lb = bounds[:, 0]
        self.ub = bounds[:, 1]
        self.cfg = config
        self.rng = np.random.default_rng(config.seed)

        self.best_x = None
        self.best_f = np.inf
        self.history = []
        self.FE = 0

    # ---------------- Utilities ----------------
    def _evaluate_pop(self, X):
        self.FE += len(X)
        return np.array([self.obj(x) for x in X])

    def _clip(self, X):
        return np.clip(X, self.lb, self.ub)

    def _normalize_to_bounds(self, X):
        X = (X - X.min()) / (X.max() - X.min() + 1e-12)
        return self.lb + X * (self.ub - self.lb)

    def _distance(self, a, b):
        span = np.maximum(self.ub - self.lb, 1e-12)
        d = np.linalg.norm((a - b) / span)
        return min(d / np.sqrt(self.dim), 1.0)

    # ---------------- Sowing ----------------

    def _sowing(self):
        n, d = self.cfg.pop_size, self.dim

        if self.cfg.sowing_method == "random_init":
            X = self.rng.random((n, d))
        elif self.cfg.sowing_method == "beta_init_3_2":
            X = beta.rvs(3, 2, size=(n, d), random_state=self.rng)
        elif self.cfg.sowing_method == "beta_init_2_5_2_5":
            X = beta.rvs(2.5, 2.5, size=(n, d), random_state=self.rng)
        elif self.cfg.sowing_method == "uniform_init":
            X = uniform.rvs(size=(n, d), random_state=self.rng)
        elif self.cfg.sowing_method == "lognorm_init_0_0_5":
            X = lognorm.rvs(s=0.5, size=(n, d), random_state=self.rng)
        elif self.cfg.sowing_method == "lognorm_init_0_0_67":
            X = lognorm.rvs(s=0.67, size=(n, d), random_state=self.rng)
        elif self.cfg.sowing_method == "expon_init_0_5":
            X = expon.rvs(scale=2.0, size=(n, d), random_state=self.rng)
        elif self.cfg.sowing_method == "rayleigh_init_0_4":
            X = rayleigh.rvs(scale=0.4, size=(n, d), random_state=self.rng)
        elif self.cfg.sowing_method == "weibull_init_1_1":
            X = weibull_min.rvs(1, scale=1, size=(n, d), random_state=self.rng)
        elif self.cfg.sowing_method == "lhs_init":
            sampler = qmc.LatinHypercube(d=d, seed=self.rng)
            X = sampler.random(n)
        else:
            raise ValueError("Unknown sowing method")

        return self._normalize_to_bounds(X)

    # ---------------- Operators ----------------

    def _mating(self, G):
        idx = self.rng.permutation(len(G))
        children = []

        for i in range(0, len(idx) - 1, 2):
            a, b = G[idx[i]], G[idx[i + 1]]
            d = self._distance(a, b)
            p = np.exp(-self.cfg.alpha_mating * d)
            if self.rng.random() <= p:
                w = self.rng.random(self.dim)
                children.append(self._clip(w * a + (1 - w) * b))

        children = np.array(children)
        # **YENİ KONTROL EKLE:** Boşsa, 0 satırlı, D sütunlu bir dizi döndür.
        if children.size == 0:
            return np.empty((0, self.dim))
    
        return children

    def _branching(self, G):
        n = len(G)
        m = max(1, int(self.cfg.branching_rate * n))
        idxs = self.rng.choice(n, size=m, replace=False)
        kids = []

        for i in idxs:
            x = G[i].copy()
            j = self.rng.integers(0, self.dim)
            span = self.ub[j] - self.lb[j]
            x[j] += self.rng.normal(0, self.cfg.branch_sigma * span)
            kids.append(self._clip(x))

        kids = np.array(kids)
        # **YENİ KONTROL EKLE:**
        if kids.size == 0:
            return np.empty((0, self.dim))
        
        return kids

    def _vaccinating(self, G, F):
        n = len(G)
        m = max(1, int(self.cfg.vaccination_rate * n))
        order = np.argsort(F)
        best = order[:n // 2]
        worst = order[n // 2:]
        kids = []

        for i in self.rng.choice(worst, size=m, replace=False):
            donor = G[self.rng.choice(best)]
            kids.append(self._clip(G[i] + self.cfg.vacc_intensity * (donor - G[i])))

        kids = np.array(kids)
        # **YENİ KONTROL EKLE:**
        if kids.size == 0:
            return np.empty((0, self.dim))
        
        return kids

    # ---------------- Run ----------------

    def run(self):
        G = self._sowing()
        F = self._evaluate_pop(G)

        self.best_f = F.min()
        self.best_x = G[F.argmin()].copy()
        self.history = [self.best_f]

        for _ in range(self.cfg.max_iters):
            G_all = np.vstack([G, self._mating(G), self._branching(G), self._vaccinating(G, F)])
            F_all = self._evaluate_pop(G_all)
            order = np.argsort(F_all)
            G = G_all[order[:self.cfg.pop_size]]
            F = F_all[order[:self.cfg.pop_size]]

            if F[0] < self.best_f:
                self.best_f = F[0]
                self.best_x = G[0].copy()

            self.history.append(self.best_f)

        return {"best_x": self.best_x, "best_f": self.best_f, "history": np.array(self.history),"FE": self.FE}

# =========================================================
# CEC FUNCTIONS (F1–F11)
# =========================================================
def f3_optimized(x):
    # np.cumsum(x): x'in kümülatif toplamını tek bir hızlı NumPy işlemiyle hesaplar.
    # Örneğin, [a, b, c] -> [a, a+b, a+b+c]
    cumulative_sums = np.cumsum(x)
    
    # Kümülatif toplamların karesini alıp toplar.
    return np.sum(cumulative_sums**2)
    
def u(x, a, k, m):
    return np.where(
        x > a, k * (x - a)**m,
        np.where(x < -a, k * (-x - a)**m, 0)
    )
def get_cec_function_details(D=10):
    functions = {}

    functions["F1"] = (lambda x: np.sum(x**2),
        np.column_stack([np.full(D, -100.0), np.full(D, 100.0)]))

    functions["F2"] = (lambda x: np.sum(np.abs(x)) + np.prod(np.abs(x)),
        np.column_stack([np.full(D, -10.0), np.full(D, 10.0)]))

    functions["F3"] = (
        f3_optimized,
        np.column_stack([np.full(D, -100.0), np.full(D, 100.0)]))

    functions["F4"] = (lambda x: np.max(np.abs(x)),
        np.column_stack([np.full(D, -100.0), np.full(D, 100.0)]))

    functions["F5"] = (lambda x: np.sum(100*(x[1:] - x[:-1]**2)**2 + (x[:-1] - 1)**2),
        np.column_stack([np.full(D, -30.0), np.full(D, 30.0)]))

    functions["F6"] = (lambda x: np.sum((np.floor(x) + 0.5)**2),
        np.column_stack([np.full(D, -100.0), np.full(D, 100.0)]))

    functions["F7"] = (lambda x: np.sum(np.arange(1, len(x)+1) * x**4),
        np.column_stack([np.full(D, -1.28), np.full(D, 1.28)]))

    functions["F8"] = (lambda x: -np.sum(x * np.sin(np.sqrt(np.abs(x)))),
        np.column_stack([np.full(D, -500.0), np.full(D, 500.0)]))

    functions["F9"] = (lambda x: np.sum(x**2 - 10*np.cos(2*np.pi*x) + 10),
        np.column_stack([np.full(D, -5.12), np.full(D, 5.12)]))

    functions["F10"] = (lambda x: -20*np.exp(-0.2*np.sqrt(np.mean(x**2)))
        - np.exp(np.mean(np.cos(2*np.pi*x))) + 20 + np.e,
        np.column_stack([np.full(D, -32.0), np.full(D, 32.0)]))

    functions["F11"] = (lambda x: np.sum(x**2)/4000 - np.prod(np.cos(x/np.sqrt(np.arange(1,len(x)+1)))) + 1,
        np.column_stack([np.full(D, -600.0), np.full(D, 600.0)]))
    # Penalized 1
    def penalized_1(x):
        n = len(x)
        y = 1 + (x + 1) / 4
        term1 = np.pi / n * (
            10 * np.sin(np.pi * y[0])**2 +
            np.sum((y[:-1] - 1)**2 * (1 + 10 * np.sin(np.pi * y[1:])**2)) +
            (y[-1] - 1)**2
        )
        term2 = np.sum(u(x, 10, 100, 4))
        return term1 + term2

    bounds_p1 = np.column_stack([np.full(D, -50.0), np.full(D, 50.0)])
    functions["F12: Penalized 1"] = (penalized_1, bounds_p1)

    # Penalized 2
    def penalized_2(x):
        n = len(x)
        term1 = 0.1 * (
            np.sin(3 * np.pi * x[0])**2 +
            np.sum((x[:-1] - 1)**2 * (1 + np.sin(3 * np.pi * x[1:])**2)) +
            (x[-1] - 1)**2 * (1 + np.sin(2 * np.pi * x[-1])**2)
        )
        term2 = np.sum(u(x, 5, 100, 4))
        return term1 + term2

    bounds_p2 = np.column_stack([np.full(D, -50.0), np.full(D, 50.0)])
    functions["F13: Penalized 2"] = (penalized_2, bounds_p2)
    return functions

def get_fixed_cec_functions():
    functions = {}

    # ======================================================
    # Shekel's Foxholes (2D)
    # ======================================================
    def shekel(x):
        a = np.array([
            [-32, -16, 0, 16, 32] * 5,
            [-32]*5 + [-16]*5 + [0]*5 + [16]*5 + [32]*5
        ])
        c = np.arange(1, 26)
        return np.sum(1.0 / (c + np.sum((x[:, None] - a) ** 6, axis=0)))

    bounds = np.array([[-65, 65], [-65, 65]])
    functions["Shekel"] = (shekel, bounds, 2)

    # ======================================================
    # Branin (2D)
    # ======================================================
    def branin(x):
        x1, x2 = x
        a = 1.0
        b = 5.1 / (4 * np.pi**2)
        c = 5 / np.pi
        r = 6
        s = 10
        t = 1 / (8 * np.pi)
        return a * (x2 - b*x1**2 + c*x1 - r)**2 + s*(1 - t)*np.cos(x1) + s

    bounds = np.array([[-5, 5], [0, 15]])
    functions["Branin"] = (branin, bounds, 2)

    # ======================================================
    # Six-Hump Camel (2D)
    # ======================================================
    def six_hump_camel(x):
        x1, x2 = x
        return (
            (4 - 2.1*x1**2 + (x1**4)/3)*x1**2 +
            x1*x2 +
            (-4 + 4*x2**2)*x2**2
        )

    bounds = np.array([[-5, 5], [-5, 5]])
    functions["Six-Hump Camel"] = (six_hump_camel, bounds, 2)

    # ======================================================
    # Goldstein-Price (2D)
    # ======================================================
    def goldstein_price(x):
        x1, x2 = x
        term1 = 1 + (x1 + x2 + 1)**2 * (
            19 - 14*x1 + 3*x1**2 - 14*x2 + 6*x1*x2 + 3*x2**2
        )
        term2 = 30 + (2*x1 - 3*x2)**2 * (
            18 - 32*x1 + 12*x1**2 + 48*x2 - 36*x1*x2 + 27*x2**2
        )
        return term1 * term2

    bounds = np.array([[-2, 2], [-2, 2]])
    functions["Goldstein-Price"] = (goldstein_price, bounds, 2)

    # ======================================================
    # Kowalik's Function (4D)
    # ======================================================
    def kowalik(x):
        a = np.array([
            0.1957, 0.1947, 0.1735, 0.16,
            0.0844, 0.0627, 0.0456, 0.0342,
            0.0323, 0.0235, 0.0246
        ])
        b = 1.0 / np.array([
            0.25, 0.5, 1, 2, 4, 6,
            8, 10, 12, 14, 16
        ])
        return np.sum((a - (x[0]*(b**2 + b*x[1]) /
                (b**2 + b*x[2] + x[3])))**2)

    bounds = np.array([[-5, 5]] * 4)
    functions["Kowalik"] = (kowalik, bounds, 4)

    # ======================================================
    # Hartman 3D
    # ======================================================
    def hartman3(x):
        alpha = np.array([1.0, 1.2, 3.0, 3.2])
        A = np.array([
            [3.0, 10, 30],
            [0.1, 10, 35],
            [3.0, 10, 30],
            [0.1, 10, 35]
        ])
        P = 1e-4 * np.array([
            [3689, 1170, 2673],
            [4699, 4387, 7470],
            [1091, 8732, 5547],
            [381, 5743, 8828]
        ])
        return -np.sum(alpha * np.exp(-np.sum(A * (x - P)**2, axis=1)))

    bounds = np.array([[0, 1]] * 3)
    functions["Hartman 3D"] = (hartman3, bounds, 3)

    # ======================================================
    # Hartman 6D
    # ======================================================
    def hartman6(x):
        alpha = np.array([1.0, 1.2, 3.0, 3.2])
        A = np.array([
            [10, 3, 17, 3.5, 1.7, 8],
            [0.05, 10, 17, 0.1, 8, 14],
            [3, 3.5, 1.7, 10, 17, 8],
            [17, 8, 0.05, 10, 0.1, 14]
        ])
        P = 1e-4 * np.array([
            [1312, 1696, 5569, 124, 8283, 5886],
            [2329, 4135, 8307, 3736, 1004, 9991],
            [2348, 1451, 3522, 2883, 3047, 6650],
            [4047, 8828, 8732, 5743, 1091, 381]
        ])
        return -np.sum(alpha * np.exp(-np.sum(A * (x - P)**2, axis=1)))

    bounds = np.array([[0, 1]] * 6)
    functions["Hartman 6D"] = (hartman6, bounds, 6)

    return functions
# =========================================================
# EXPERIMENT RUNNER
# =========================================================
def run_fixed_cec_tests(N_RUNS=30, MAX_ITERS=500, POP_SIZE=50):

    functions = get_fixed_cec_functions()

    print("\n=== Fixed-Dimension CEC Functions ===")
    print("Function        Sowing              Best        Mean        Std")

    for fname, (fn, bounds, dim) in functions.items():

        for sowing_method in SOWING_METHODS:

            scores = []

            for r in range(N_RUNS):
                cfg = SGuAConfig(
                    pop_size=POP_SIZE,
                    max_iters=MAX_ITERS,
                    sowing_method=sowing_method,
                    seed=r
                )

                algo = SGuA(fn, dim, bounds, cfg)
                res = algo.run()
                scores.append(res["best_f"])

            scores = np.array(scores)

            print(
                f"{fname:<15} "
                f"{sowing_method:<18} "
                f"{scores.min():.3e} "
                f"{scores.mean():.3e} "
                f"{scores.std():.3e}"
            )

def run_cec_tests(D=10, N_RUNS=30, MAX_ITERS=500, POP_SIZE=50, sowing_methods=None):

    if sowing_methods is None:
        sowing_methods = ["random_init", "beta_init_3_2", "lhs_init"]

    functions = get_cec_function_details(D)

    print("Function  Sowing              Best        Mean        Std         FE")

    for name, (fn, bounds) in functions.items():

        all_scores = {}   # sowing -> scores
        all_fes = {}

        for sowing in sowing_methods:

            cfg = SGuAConfig(
                pop_size=POP_SIZE,
                max_iters=MAX_ITERS,
                sowing_method=sowing
            )

            scores = []
            fes = []

            for r in range(N_RUNS):
                cfg.seed = r
                algo = SGuA(fn, D, bounds, cfg)
                res = algo.run()
                scores.append(res["best_f"])
                fes.append(res["FE"])

            scores = np.array(scores)
            all_scores[sowing] = scores
            all_fes[sowing] = np.mean(fes)

            print(
                f"{name:<8} "
                f"{sowing:<18} "
                f"{scores.min():.3e} "
                f"{scores.mean():.3e} "
                f"{scores.std():.3e} "
                f"{all_fes[sowing]:.0f}"
            )

        # ===== WILCOXON TAM BURADA =====
        print(f"\nWilcoxon results for {name}:")
        pairs = list(combinations(comparison_sowing_methods, 2))

        # Only perform Wilcoxon tests if all methods for comparison were run
        if all(method in all_scores for method in [p for pair in pairs for p in pair]):
            for a, b in pairs:
                stat, p = wilcoxon(all_scores[a], all_scores[b])
                print(f"  {a} vs {b} : p = {p:.4f}")
        else:
            print("  Not enough sowing methods were run to perform all Wilcoxon comparisons.")

        print("-" * 70)
# =========================================================
# MAIN
# =========================================================

if __name__ == "__main__":

    DIMENSION = 50
    RUNS = 30
    ITERS = 2500
    POP = 50

    # Run CEC tests with all comparison methods in one go
    comparison_sowing_methods = [
         "random_init",
        "beta_init_3_2",
        "beta_init_2_5_2_5",
        "uniform_init",
        "lognorm_init_0_0_5",
        "lognorm_init_0_0_67",
        "expon_init_0_5",
        "rayleigh_init_0_4",
        "weibull_init_1_1",
        "lhs_init"
    ]
    print(f"\n=== Running CEC tests with multiple sowing methods for comparison ===")
    run_cec_tests(
        D=DIMENSION,
        N_RUNS=RUNS,
        MAX_ITERS=ITERS,
        POP_SIZE=POP,
        sowing_methods=comparison_sowing_methods
    )

    print("\n=== Fixed-dimension CEC tests ===")
    run_fixed_cec_tests(
        N_RUNS=RUNS,
        MAX_ITERS=ITERS,
        POP_SIZE=POP
    )


=== Running CEC tests with multiple sowing methods for comparison ===
Function  Sowing              Best        Mean        Std         FE
F1       random_init        3.979e-02 1.153e-01 3.901e-02 279794
F1       beta_init_3_2      5.463e-02 1.300e-01 4.381e-02 279746
F1       beta_init_2_5_2_5  4.730e-02 1.090e-01 4.093e-02 279854
F1       uniform_init       3.979e-02 1.153e-01 3.901e-02 279794
F1       lognorm_init_0_0_5 1.058e-01 1.806e-01 5.089e-02 279223
F1       lognorm_init_0_0_67 9.476e-02 1.966e-01 6.177e-02 279075
F1       expon_init_0_5     6.708e-02 1.933e-01 6.633e-02 279088
F1       rayleigh_init_0_4  8.669e-02 1.637e-01 4.659e-02 279524
F1       weibull_init_1_1   9.883e-02 1.852e-01 5.642e-02 279095
F1       lhs_init           4.243e-02 1.262e-01 5.746e-02 279815

Wilcoxon results for F1:
  random_init vs beta_init_3_2 : p = 0.1004
  random_init vs beta_init_2_5_2_5 : p = 0.3285
  random_init vs uniform_init : p = nan
  random_init vs lognorm_init_0_0_5 : p = 0.0000
  

d=200, f1-f8

In [7]:
import numpy as np
from dataclasses import dataclass
from typing import Callable, Tuple, Optional, Dict, Any, List
from scipy.stats import beta, uniform, lognorm, expon, rayleigh, weibull_min
from scipy.stats import qmc
from scipy.stats import wilcoxon
from itertools import combinations

# =========================================================
# CONFIG
# =========================================================

@dataclass
class SGuAConfig:
    mode: str = "continuous"
    pop_size: int = 50
    max_iters: int = 500
    minimize: bool = True
    sowing_method: str = "random_init"
    mating_model: str = "exponential"
    branching_model: str = "linear"
    vaccination_rate: float = 0.25
    branching_rate: float = 0.5
    mating_pairs: str = "random"
    alpha_mating: float = 4.0
    branch_sigma: float = 0.15
    vacc_intensity: float = 0.5
    seed: Optional[int] = None
    f_target: Optional[float] = None

# =========================================================
# SGuA
# =========================================================
SOWING_METHODS = [
    "random_init",
    "beta_init_3_2",
    "beta_init_2_5_2_5",
    "uniform_init",
    "lognorm_init_0_0_5",
    "lognorm_init_0_0_67",
    "expon_init_0_5",
    "rayleigh_init_0_4",
    "weibull_init_1_1",
    "lhs_init"
]
class SGuA:

    def __init__(self, objective, dim, bounds, config: SGuAConfig):
        self.obj = objective
        self.dim = dim
        self.lb = bounds[:, 0]
        self.ub = bounds[:, 1]
        self.cfg = config
        self.rng = np.random.default_rng(config.seed)

        self.best_x = None
        self.best_f = np.inf
        self.history = []
        self.FE = 0

    # ---------------- Utilities ----------------
    def _evaluate_pop(self, X):
        self.FE += len(X)
        return np.array([self.obj(x) for x in X])

    def _clip(self, X):
        return np.clip(X, self.lb, self.ub)

    def _normalize_to_bounds(self, X):
        X = (X - X.min()) / (X.max() - X.min() + 1e-12)
        return self.lb + X * (self.ub - self.lb)

    def _distance(self, a, b):
        span = np.maximum(self.ub - self.lb, 1e-12)
        d = np.linalg.norm((a - b) / span)
        return min(d / np.sqrt(self.dim), 1.0)

    # ---------------- Sowing ----------------

    def _sowing(self):
        n, d = self.cfg.pop_size, self.dim

        if self.cfg.sowing_method == "random_init":
            X = self.rng.random((n, d))
        elif self.cfg.sowing_method == "beta_init_3_2":
            X = beta.rvs(3, 2, size=(n, d), random_state=self.rng)
        elif self.cfg.sowing_method == "beta_init_2_5_2_5":
            X = beta.rvs(2.5, 2.5, size=(n, d), random_state=self.rng)
        elif self.cfg.sowing_method == "uniform_init":
            X = uniform.rvs(size=(n, d), random_state=self.rng)
        elif self.cfg.sowing_method == "lognorm_init_0_0_5":
            X = lognorm.rvs(s=0.5, size=(n, d), random_state=self.rng)
        elif self.cfg.sowing_method == "lognorm_init_0_0_67":
            X = lognorm.rvs(s=0.67, size=(n, d), random_state=self.rng)
        elif self.cfg.sowing_method == "expon_init_0_5":
            X = expon.rvs(scale=2.0, size=(n, d), random_state=self.rng)
        elif self.cfg.sowing_method == "rayleigh_init_0_4":
            X = rayleigh.rvs(scale=0.4, size=(n, d), random_state=self.rng)
        elif self.cfg.sowing_method == "weibull_init_1_1":
            X = weibull_min.rvs(1, scale=1, size=(n, d), random_state=self.rng)
        elif self.cfg.sowing_method == "lhs_init":
            sampler = qmc.LatinHypercube(d=d, seed=self.rng)
            X = sampler.random(n)
        else:
            raise ValueError("Unknown sowing method")

        return self._normalize_to_bounds(X)

    # ---------------- Operators ----------------

    def _mating(self, G):
        idx = self.rng.permutation(len(G))
        children = []

        for i in range(0, len(idx) - 1, 2):
            a, b = G[idx[i]], G[idx[i + 1]]
            d = self._distance(a, b)
            p = np.exp(-self.cfg.alpha_mating * d)
            if self.rng.random() <= p:
                w = self.rng.random(self.dim)
                children.append(self._clip(w * a + (1 - w) * b))

        children = np.array(children)
        # **YENİ KONTROL EKLE:** Boşsa, 0 satırlı, D sütunlu bir dizi döndür.
        if children.size == 0:
            return np.empty((0, self.dim))
    
        return children

    def _branching(self, G):
        n = len(G)
        m = max(1, int(self.cfg.branching_rate * n))
        idxs = self.rng.choice(n, size=m, replace=False)
        kids = []

        for i in idxs:
            x = G[i].copy()
            j = self.rng.integers(0, self.dim)
            span = self.ub[j] - self.lb[j]
            x[j] += self.rng.normal(0, self.cfg.branch_sigma * span)
            kids.append(self._clip(x))

        kids = np.array(kids)
        # **YENİ KONTROL EKLE:**
        if kids.size == 0:
            return np.empty((0, self.dim))
        
        return kids

    def _vaccinating(self, G, F):
        n = len(G)
        m = max(1, int(self.cfg.vaccination_rate * n))
        order = np.argsort(F)
        best = order[:n // 2]
        worst = order[n // 2:]
        kids = []

        for i in self.rng.choice(worst, size=m, replace=False):
            donor = G[self.rng.choice(best)]
            kids.append(self._clip(G[i] + self.cfg.vacc_intensity * (donor - G[i])))

        kids = np.array(kids)
        # **YENİ KONTROL EKLE:**
        if kids.size == 0:
            return np.empty((0, self.dim))
        
        return kids

    # ---------------- Run ----------------

    def run(self):
        G = self._sowing()
        F = self._evaluate_pop(G)

        self.best_f = F.min()
        self.best_x = G[F.argmin()].copy()
        self.history = [self.best_f]

        for _ in range(self.cfg.max_iters):
            G_all = np.vstack([G, self._mating(G), self._branching(G), self._vaccinating(G, F)])
            F_all = self._evaluate_pop(G_all)
            order = np.argsort(F_all)
            G = G_all[order[:self.cfg.pop_size]]
            F = F_all[order[:self.cfg.pop_size]]

            if F[0] < self.best_f:
                self.best_f = F[0]
                self.best_x = G[0].copy()

            self.history.append(self.best_f)

        return {"best_x": self.best_x, "best_f": self.best_f, "history": np.array(self.history),"FE": self.FE}

# =========================================================
# CEC FUNCTIONS (F1–F11)
# =========================================================
def f3_optimized(x):
    # np.cumsum(x): x'in kümülatif toplamını tek bir hızlı NumPy işlemiyle hesaplar.
    # Örneğin, [a, b, c] -> [a, a+b, a+b+c]
    cumulative_sums = np.cumsum(x)
    
    # Kümülatif toplamların karesini alıp toplar.
    return np.sum(cumulative_sums**2)
    
def u(x, a, k, m):
    return np.where(
        x > a, k * (x - a)**m,
        np.where(x < -a, k * (-x - a)**m, 0)
    )
    
def get_cec_function_details(D=10):
    functions = {}

    

    functions["F8"] = (lambda x: -np.sum(x * np.sin(np.sqrt(np.abs(x)))),
        np.column_stack([np.full(D, -500.0), np.full(D, 500.0)]))

    functions["F9"] = (lambda x: np.sum(x**2 - 10*np.cos(2*np.pi*x) + 10),
        np.column_stack([np.full(D, -5.12), np.full(D, 5.12)]))

    functions["F10"] = (lambda x: -20*np.exp(-0.2*np.sqrt(np.mean(x**2)))
        - np.exp(np.mean(np.cos(2*np.pi*x))) + 20 + np.e,
        np.column_stack([np.full(D, -32.0), np.full(D, 32.0)]))

    functions["F11"] = (lambda x: np.sum(x**2)/4000 - np.prod(np.cos(x/np.sqrt(np.arange(1,len(x)+1)))) + 1,
        np.column_stack([np.full(D, -600.0), np.full(D, 600.0)]))

    functions["F7_Makale"] = (lambda x: -np.sum(np.sin(x) * np.sin((np.arange(1, len(x) + 1) * x**2) / np.pi)**20),
        np.column_stack([np.full(D, 0.0), np.full(D, np.pi)]))

    functions["F8_Makale"] = (lambda x: np.mean(x**4 - 16*x**2 + 5*x),
        np.column_stack([np.full(D, -5.0), np.full(D, 5.0)]))
    
    # Penalized 1
    def penalized_1(x):
        n = len(x)
        y = 1 + (x + 1) / 4
        term1 = np.pi / n * (
            10 * np.sin(np.pi * y[0])**2 +
            np.sum((y[:-1] - 1)**2 * (1 + 10 * np.sin(np.pi * y[1:])**2)) +
            (y[-1] - 1)**2
        )
        term2 = np.sum(u(x, 10, 100, 4))
        return term1 + term2

    bounds_p1 = np.column_stack([np.full(D, -50.0), np.full(D, 50.0)])
    functions["F12: Penalized 1"] = (penalized_1, bounds_p1)

    # Penalized 2
    def penalized_2(x):
        n = len(x)
        term1 = 0.1 * (
            np.sin(3 * np.pi * x[0])**2 +
            np.sum((x[:-1] - 1)**2 * (1 + np.sin(3 * np.pi * x[1:])**2)) +
            (x[-1] - 1)**2 * (1 + np.sin(2 * np.pi * x[-1])**2)
        )
        term2 = np.sum(u(x, 5, 100, 4))
        return term1 + term2

    bounds_p2 = np.column_stack([np.full(D, -50.0), np.full(D, 50.0)])
    functions["F13: Penalized 2"] = (penalized_2, bounds_p2)
    return functions


# =========================================================
# EXPERIMENT RUNNER
# =========================================================

def run_cec_tests(D=10, N_RUNS=30, MAX_ITERS=500, POP_SIZE=50, sowing_methods=None):

    if sowing_methods is None:
        sowing_methods = ["random_init", "beta_init_3_2", "lhs_init"]

    functions = get_cec_function_details(D)

    print("Function  Sowing              Best        Mean        Std         FE")

    for name, (fn, bounds) in functions.items():

        all_scores = {}   # sowing -> scores
        all_fes = {}

        for sowing in sowing_methods:

            cfg = SGuAConfig(
                pop_size=POP_SIZE,
                max_iters=MAX_ITERS,
                sowing_method=sowing
            )

            scores = []
            fes = []

            for r in range(N_RUNS):
                cfg.seed = r
                algo = SGuA(fn, D, bounds, cfg)
                res = algo.run()
                scores.append(res["best_f"])
                fes.append(res["FE"])

            scores = np.array(scores)
            all_scores[sowing] = scores
            all_fes[sowing] = np.mean(fes)

            print(
                f"{name:<8} "
                f"{sowing:<18} "
                f"{scores.min():.3e} "
                f"{scores.mean():.3e} "
                f"{scores.std():.3e} "
                f"{all_fes[sowing]:.0f}"
            )

        # ===== WILCOXON TAM BURADA =====
        print(f"\nWilcoxon results for {name}:")
        pairs = list(combinations(comparison_sowing_methods, 2))

        # Only perform Wilcoxon tests if all methods for comparison were run
        if all(method in all_scores for method in [p for pair in pairs for p in pair]):
            for a, b in pairs:
                stat, p = wilcoxon(all_scores[a], all_scores[b])
                print(f"  {a} vs {b} : p = {p:.4f}")
        else:
            print("  Not enough sowing methods were run to perform all Wilcoxon comparisons.")

        print("-" * 70)
# =========================================================
# MAIN
# =========================================================

if __name__ == "__main__":

    DIMENSION = 200
    RUNS = 30
    ITERS = 3500
    POP = 100

    # Run CEC tests with all comparison methods in one go
    comparison_sowing_methods = [
         "random_init",
        "beta_init_3_2",
        "beta_init_2_5_2_5",
        "uniform_init",
        "lognorm_init_0_0_5",
        "lognorm_init_0_0_67",
        "expon_init_0_5",
        "rayleigh_init_0_4",
        "weibull_init_1_1",
        "lhs_init"
    ]
    print(f"\n=== Running CEC tests with multiple sowing methods for comparison ===")
    run_cec_tests(
        D=DIMENSION,
        N_RUNS=RUNS,
        MAX_ITERS=ITERS,
        POP_SIZE=POP,
        sowing_methods=comparison_sowing_methods
    )


=== Running CEC tests with multiple sowing methods for comparison ===
Function  Sowing              Best        Mean        Std         FE
F8       random_init        -7.364e+04 -7.247e+04 7.122e+02 780483
F8       beta_init_3_2      -7.717e+04 -7.573e+04 6.916e+02 778719
F8       beta_init_2_5_2_5  -7.349e+04 -7.205e+04 7.692e+02 778302
F8       uniform_init       -7.364e+04 -7.247e+04 7.122e+02 780483
F8       lognorm_init_0_0_5 -6.018e+04 -6.009e+04 4.807e+01 786532
F8       lognorm_init_0_0_67 -6.041e+04 -6.011e+04 1.059e+02 782570
F8       expon_init_0_5     -6.064e+04 -6.029e+04 1.502e+02 782140
F8       rayleigh_init_0_4  -6.159e+04 -6.046e+04 3.620e+02 784302
F8       weibull_init_1_1   -6.087e+04 -6.024e+04 1.846e+02 782064
F8       lhs_init           -7.400e+04 -7.233e+04 8.815e+02 780615

Wilcoxon results for F8:
  random_init vs beta_init_3_2 : p = 0.0000
  random_init vs beta_init_2_5_2_5 : p = 0.0636
  random_init vs uniform_init : p = nan
  random_init vs lognorm_init_0